# SpectralKAN — Full Pipeline Notebook

End-to-end driver for the SpectralKAN project (interpretable masked autoencoding for biosignals via a KAN decoder).

Every stage of the project is controllable from this notebook:

1. **Environment check** — verify Python / PyTorch / CUDA and the project layout.
2. **Config selection** — choose dataset (`esc50` / `ptbxl`), decoder (`transformer` / `kan`), and runtime overrides.
3. **Smoke test** — build the model and run a tiny forward pass to catch shape / import problems before a long run.
4. **Pretraining** — call `scripts/train.py` with the resolved config + overrides.
5. **Run inspection** — list result folders, plot training-loss / band curves.
6. **Linear-probe + k-NN evaluation** — call `scripts/evaluate.py` on the chosen checkpoint.
7. **Frequency-band analysis (Claim 1)** — call `scripts/freq_band_analysis.py` to compare transformer vs KAN reconstructions.

## How to use

* **Order**: run cells top-to-bottom the first time. After that, the per-stage cells are independent — re-run any individual stage as long as `Section 1` (paths) and `Section 2` (knobs) have been executed.
* **Knobs**: `Section 2` is the *only* place you should normally edit. All later cells read from those variables, so changing the dataset / decoder / epoch count there propagates to training, evaluation, and analysis without manual edits below.
* **Subprocess vs in-process**: training and evaluation shell out to the existing scripts (`scripts/train.py`, `scripts/evaluate.py`, `scripts/freq_band_analysis.py`) so the notebook stays a thin driver — what you run from a notebook cell is byte-identical to running from the terminal. Smoke checks and run-inspection use the project's Python modules directly.
* **Long runs**: a real ESC-50 / PTB-XL pretrain takes hours on a single GPU. Set `EPOCHS` to something small (e.g. 2–5) the first time to confirm the pipeline before launching the full job.
* **Hardware**: per `CLAUDE.md`, keep `BATCH_SIZE ≤ 128` on the RTX 3060 and `≤ 256` on the RTX 4070 Ti Super.

## Section 1 — Project paths & imports

Locate the project root, switch CWD to it (so the relative paths inside the YAML configs resolve correctly), and add the project root to `sys.path`. Run this cell once before anything else.

In [ ]:
import os, sys
from pathlib import Path

# notebook lives at <project_root>/notebooks/run_pipeline.ipynb
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

# Make project packages importable just like the scripts do.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"CWD          = {Path.cwd()}")
for required in ("configs", "data", "models", "scripts", "results"):
    p = PROJECT_ROOT / required
    print(f"  {required:<10} -> {'OK' if p.exists() else 'MISSING'} ({p})")

## Section 1b — Environment / GPU check

Quick sanity check on the runtime. If `cuda.is_available() == False`, training will still work on CPU but extremely slowly — the rest of the notebook will still function.

In [ ]:
import platform
import torch

print(f"Python      : {platform.python_version()}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA build  : {torch.version.cuda}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    idx = torch.cuda.current_device()
    print(f"GPU         : {torch.cuda.get_device_name(idx)}")
    props = torch.cuda.get_device_properties(idx)
    print(f"VRAM        : {props.total_memory / 1024**3:.1f} GiB")

## Section 2 — Pipeline knobs (edit me)

These are the only variables you should normally edit. Everything below reads from them.

* `DATASET`            — `"esc50"` (audio) or `"ptbxl"` (ECG).
* `DECODER`            — `"transformer"` (baseline) or `"kan"` (our contribution).
* `EPOCHS`             — total pretraining epochs. Use `2`–`5` for a smoke run, `200` for the full recipe.
* `BATCH_SIZE`         — per-step batch. RTX 3060 ≤ 128, RTX 4070 Ti Super ≤ 256.
* `NUM_WORKERS`        — DataLoader workers. `0` if you hit Windows fork issues; `4` is a good default.
* `SAVE_EVERY`         — checkpoint cadence (numbered). `last.pt` is always written every epoch regardless.
* `SEED`               — RNG seed for reproducibility.
* `DEVICE`             — `"cuda"` or `"cpu"`. Auto-falls-back to CPU if CUDA is missing.
* `EXTRA_TRAIN_ARGS`   — list of extra CLI args appended verbatim to `scripts/train.py` (e.g. `["--resume", "results/<run>/checkpoints/last.pt"]`).

In [ ]:
DATASET     = "esc50"        # "esc50" | "ptbxl"
DECODER     = "transformer"  # "transformer" | "kan"

EPOCHS      = 5              # use 200 for the full recipe
BATCH_SIZE  = 64
NUM_WORKERS = 4
SAVE_EVERY  = 20
SEED        = 42
DEVICE      = "cuda"         # "cuda" | "cpu"

EXTRA_TRAIN_ARGS = []

assert DATASET in ("esc50", "ptbxl"), DATASET
assert DECODER in ("transformer", "kan"), DECODER

# Pick the right base config for the (dataset, decoder) pair.
if DECODER == "kan":
    CONFIG_PATH = PROJECT_ROOT / "configs" / f"{DATASET}_kan_config.yaml"
else:
    CONFIG_PATH = PROJECT_ROOT / "configs" / f"{DATASET}_config.yaml"

assert CONFIG_PATH.exists(), f"missing config: {CONFIG_PATH}"
print(f"Using config: {CONFIG_PATH.relative_to(PROJECT_ROOT)}")

## Section 3 — Resolve the merged config

`train.load_config` follows the `inherits:` chain (deep-merge, child wins). This cell shows you the *final* config dict that the training loop will see — useful for catching typos in the YAML before launching a long run.

In [ ]:
import yaml
from utils.setup import load_config, bridge_config

raw_cfg = load_config(CONFIG_PATH)
in_chans = 1 if DATASET == "esc50" else 12
bridged_cfg = bridge_config(raw_cfg, in_chans=in_chans)

print("=== merged YAML config ===")
print(yaml.safe_dump(raw_cfg, sort_keys=False))
print("=== bridged build_mae kwargs ===")
print(yaml.safe_dump(bridged_cfg, sort_keys=False))

## Section 4 — Smoke test (build model + dummy forward)

Constructs the MAE from the bridged config and runs a single forward pass on a synthetic spectrogram. This catches dimension / decoder / KAN-library issues in seconds, before you commit to a long training run.

It does **not** touch the real datasets, so it'll succeed even when ESC-50 / PTB-XL aren't downloaded yet.

In [ ]:
from models.mae import build_mae, MaskedAutoencoder

smoke_model = build_mae(bridged_cfg)
n_params = sum(p.numel() for p in smoke_model.parameters() if p.requires_grad)

# Spectrogram-shaped fake input. (B, C, F, T) — 128x128 is small enough for CPU.
fake = torch.randn(2, in_chans, 128, 128)
out = smoke_model(fake)
recon, mask, grid = smoke_model.reconstruct(fake)
bands = MaskedAutoencoder.compute_frequency_band_loss(fake, recon)

print(f"params       : {n_params:,}")
print(f"input shape  : {tuple(fake.shape)}")
print(f"loss         : {out['loss'].item():.4f}")
print(f"masked frac  : {out['mask'].float().mean().item():.3f}  (target ~0.75)")
print(f"recon shape  : {tuple(recon.shape)}  (grid={grid})")
print(f"band MSE     : low={bands['low'].item():.4f}  mid={bands['mid'].item():.4f}  high={bands['high'].item():.4f}")

# Free the smoke-test model — the real training cell builds its own.
del smoke_model, out, recon, mask, fake
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Section 5 — Pretrain (calls `scripts/train.py`)

Runs the canonical training loop in a subprocess. The CLI is built from the knobs in Section 2, so this cell is reproducible from a terminal — copy the printed command if you want to launch the same run from a shell.

Output streams into the cell. Each epoch the loop appends a row to `results/<run>/training_log.csv` and overwrites `results/<run>/checkpoints/last.pt`; numbered checkpoints (`epoch_0020.pt`, …) are written every `SAVE_EVERY` epochs.

In [ ]:
import shlex, subprocess, sys as _sys

train_cmd = [
    _sys.executable, "scripts/train.py",
    "--config", str(CONFIG_PATH),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--save-every", str(SAVE_EVERY),
    "--seed", str(SEED),
    "--device", DEVICE,
    "--decoder-type", DECODER,
    *EXTRA_TRAIN_ARGS,
]
print("$ " + " ".join(shlex.quote(c) for c in train_cmd))

# Stream stdout line-by-line so tqdm progress shows up live in the notebook.
proc = subprocess.Popen(
    train_cmd, cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
try:
    for line in proc.stdout:
        print(line, end="")
finally:
    proc.wait()
print(f"\n[train.py exited with code {proc.returncode}]")

## Section 6 — Inspect training runs

Lists every run folder under `results/`, plots the training-loss + per-band-MSE curves from the most recently modified run, and remembers its `last.pt` as `LATEST_CKPT` for the evaluation step below.

Re-run this cell after each training run to refresh the plot and `LATEST_CKPT`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results_dir = PROJECT_ROOT / "results"
run_dirs = sorted(
    [p for p in results_dir.iterdir() if p.is_dir() and (p / "training_log.csv").exists()],
    key=lambda p: p.stat().st_mtime,
) if results_dir.exists() else []

if not run_dirs:
    print("No completed runs yet — train.py must have written results/<run>/training_log.csv first.")
    LATEST_RUN = None
    LATEST_CKPT = None
else:
    print("Available runs (oldest first):")
    for p in run_dirs:
        print(f"  {p.relative_to(PROJECT_ROOT)}")
    LATEST_RUN = run_dirs[-1]
    LATEST_CKPT = LATEST_RUN / "checkpoints" / "last.pt"
    print(f"\nLATEST_RUN  = {LATEST_RUN.relative_to(PROJECT_ROOT)}")
    print(f"LATEST_CKPT = {LATEST_CKPT.relative_to(PROJECT_ROOT)}  (exists={LATEST_CKPT.exists()})")

    log = pd.read_csv(LATEST_RUN / "training_log.csv")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(log["epoch"], log["train_loss"], marker="o")
    axes[0].set(xlabel="epoch", ylabel="train_loss", title="Training loss")
    axes[0].grid(linestyle=":", alpha=0.5)
    for col, label in [("band_low", "low"), ("band_mid", "mid"), ("band_high", "high")]:
        axes[1].plot(log["epoch"], log[col], marker="o", label=label)
    axes[1].set(xlabel="epoch", ylabel="band MSE", title="Per-band reconstruction MSE")
    axes[1].legend()
    axes[1].grid(linestyle=":", alpha=0.5)
    fig.tight_layout()
    plt.show()

## Section 7 — Linear probe + k-NN evaluation (calls `scripts/evaluate.py`)

Freezes the encoder and trains a single linear head plus a cosine-kNN classifier on top of the frozen features. Metrics:

* **ESC-50** (multi-class, 50 classes) → top-1 accuracy.
* **PTB-XL** (multi-label, 5 superclasses) → macro-AUROC.

Set `EVAL_CHECKPOINT` to override; defaults to `LATEST_CKPT` from Section 6. Results land in `<run>/evaluation/eval_results.csv`.

In [ ]:
EVAL_CHECKPOINT = None    # set to a Path / str to override
EVAL_LINEAR_EPOCHS = 100  # 100 is the paper recipe; drop to 20 for a quick check
EVAL_KNN_K = [1, 5]
EVAL_BATCH_SIZE = 64

ckpt_path = Path(EVAL_CHECKPOINT) if EVAL_CHECKPOINT is not None else LATEST_CKPT
assert ckpt_path is not None and ckpt_path.exists(), (
    f"checkpoint not found: {ckpt_path}. Run Section 5 first or set EVAL_CHECKPOINT manually."
)

eval_cmd = [
    _sys.executable, "scripts/evaluate.py",
    "--checkpoint", str(ckpt_path),
    "--dataset", DATASET,
    "--batch-size", str(EVAL_BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--linear-epochs", str(EVAL_LINEAR_EPOCHS),
    "--knn-k", *map(str, EVAL_KNN_K),
    "--device", DEVICE,
    "--seed", str(SEED),
]
print("$ " + " ".join(shlex.quote(c) for c in eval_cmd))

proc = subprocess.Popen(eval_cmd, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end="")
finally:
    proc.wait()
print(f"\n[evaluate.py exited with code {proc.returncode}]")

In [ ]:
# Show the evaluation CSV that evaluate.py just appended to.
from utils.output_paths import infer_run_dir_from_checkpoint

eval_csv = infer_run_dir_from_checkpoint(ckpt_path) / "evaluation" / "eval_results.csv"
if eval_csv.exists():
    print(f"reading {eval_csv.relative_to(PROJECT_ROOT)}")
    display(pd.read_csv(eval_csv))
else:
    print(f"no eval csv at {eval_csv}")

## Section 8 — Frequency-band analysis (Claim 1, calls `scripts/freq_band_analysis.py`)

Compares per-band reconstruction MSE between a transformer-decoder MAE and a KAN-decoder MAE on the test split. Set both checkpoint paths below — they should be from runs that share the same `DATASET` (the bins are derived from the spectrogram frequency axis).

Outputs (under `<run>/freq_band/` if both checkpoints share a run, otherwise a fresh `results/freq_band/<timestamp>_<dataset>/`):

* `freq_band_mse_<dataset>.csv` — per-band MSE for both decoders.
* `freq_band_mse_<dataset>_equal.png` — equal-thirds bar chart.
* `freq_band_mse_<dataset>_clinical.png` — clinical bands (PTB-XL only).

In [ ]:
# Set these to the two checkpoints you want to compare.
TRANSFORMER_CKPT = None  # e.g. PROJECT_ROOT / "results" / "<run_t>" / "checkpoints" / "last.pt"
KAN_CKPT         = None  # e.g. PROJECT_ROOT / "results" / "<run_k>" / "checkpoints" / "last.pt"

FBA_BATCH_SIZE = 64

if TRANSFORMER_CKPT is None or KAN_CKPT is None:
    print("Set TRANSFORMER_CKPT and KAN_CKPT to two checkpoints (one per decoder type)")
    print("trained on the same dataset, then re-run this cell.")
    print("\nCandidate runs you have:")
    for p in run_dirs:
        print(f"  {p.relative_to(PROJECT_ROOT)}")
else:
    fba_cmd = [
        _sys.executable, "scripts/freq_band_analysis.py",
        "--transformer-checkpoint", str(TRANSFORMER_CKPT),
        "--kan-checkpoint", str(KAN_CKPT),
        "--dataset", DATASET,
        "--batch-size", str(FBA_BATCH_SIZE),
        "--num-workers", str(NUM_WORKERS),
        "--device", DEVICE,
    ]
    print("$ " + " ".join(shlex.quote(c) for c in fba_cmd))
    proc = subprocess.Popen(fba_cmd, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end="")
    finally:
        proc.wait()
    print(f"\n[freq_band_analysis.py exited with code {proc.returncode}]")

In [ ]:
# Show the generated band-MSE plots inline (if the analysis ran).
from IPython.display import Image, display
from utils.output_paths import infer_common_run_dir

if TRANSFORMER_CKPT is not None and KAN_CKPT is not None:
    common = infer_common_run_dir([TRANSFORMER_CKPT, KAN_CKPT])
    if common is not None:
        fba_dir = common / "freq_band"
    else:
        # Fall back: just point at the most recent results/freq_band/* folder.
        fb_root = PROJECT_ROOT / "results" / "freq_band"
        candidates = sorted([p for p in fb_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime) if fb_root.exists() else []
        fba_dir = candidates[-1] if candidates else None
    if fba_dir is not None and fba_dir.exists():
        print(f"freq-band outputs in {fba_dir.relative_to(PROJECT_ROOT)}")
        for png in sorted(fba_dir.glob("*.png")):
            print(png.name)
            display(Image(filename=str(png)))
        csvs = sorted(fba_dir.glob("*.csv"))
        if csvs:
            display(pd.read_csv(csvs[-1]))
    else:
        print("no freq-band output directory found yet")

## Section 9 — Tips & troubleshooting

* **CUDA OOM** — drop `BATCH_SIZE` (3060: ≤128, 4070 Ti Super: ≤256) or `EPOCHS`, or set `DEVICE = "cpu"` to confirm the pipeline works first.
* **Resume an interrupted run** — set `EXTRA_TRAIN_ARGS = ["--resume", "results/<run>/checkpoints/last.pt"]` in Section 2 and rerun Section 5. `train.py` rehydrates optimizer + scheduler + scaler state.
* **Switch decoders for the same dataset** — flip `DECODER` between `"transformer"` and `"kan"`, rerun Section 5; you'll get two run folders to feed Section 8.
* **Windows DataLoader hangs** — set `NUM_WORKERS = 0`. Workers respawn on Windows and can stall when fork is unavailable.
* **PTB-XL not downloaded** — `data/ecg_dataset.py` defaults to `download=False`; populate `data/ptbxl/` (or pass `download=True` once via the YAML `data.download` field) before training on PTB-XL.
* **`last.pt` missing** — Section 5 didn't complete a single epoch. Check the train output for an exception; the smoke test in Section 4 is a faster way to surface model / config errors.